# Faruq-v3 — top controls paired multiseed

Jalankan satu kombinasi `ARM`/`SEED` per runtime. Pilihan: `FCT0`, `AF2R0`, `AF2R1`, atau `AF2CAL3`; seed hanya 123 atau 2026. Output ditulis langsung ke Drive, dapat resume dari `last.pt`, dan test tidak pernah diekstrak atau dibaca.


In [ ]:
ARM = 'AF2CAL3'   # fixed parallel arm
SEED = 123     # fixed parallel seed
assert ARM in {'FCT0','AF2R0','AF2R1','AF2CAL3'}
assert SEED in {123,2026}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection')
BRANCH='codex/top-controls-multiseed-confirmation'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
os.chdir(REPO)
SRC=REPO/'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
importlib.invalidate_caches()
assert (SRC/'coffee_detector/__init__.py').is_file(), SRC
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-stb-paired-confirmation-v1/val_reports/stb_capacity_paired_confirmation.json',
    'experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json',
    'experiments/faruq-v3-af2-continuation-paired-v1/val_reports/af2_continuation_paired_confirmation.json',
))
print('PROJECT:',PROJECT_ROOT)


In [ ]:
import tarfile
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'data.yaml').is_file(), DATA
assert not (DATA/'test').exists(), 'STOP: test tidak boleh tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'
assert GROUPED.is_file(), GROUPED
STB_CONFIRM=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-stb-paired-confirmation-v1/val_reports/stb_capacity_paired_confirmation.json')
AF2_CONFIRM=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-af2-igem-paired-confirmation-v1/val_reports/af2_igem_paired_confirmation.json')
CONT_CONFIRM=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-af2-continuation-paired-v1/val_reports/af2_continuation_paired_confirmation.json')
if ARM=='FCT0':
    SOURCE=require_project_artifact(PROJECT_ROOT,f'experiments/faruq-v3-stb-paired-confirmation-v1/STB1/STB1_seed{SEED}/weights/best.pt')
else:
    SOURCE=require_project_artifact(PROJECT_ROOT,f'experiments/faruq-v3-af2-igem-paired-confirmation-v1/AF2/AF2_seed{SEED}/weights/best.pt')
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-top-controls-paired-confirmation-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
print('ARM/SEED:',ARM,SEED); print('SOURCE:',SOURCE); print('OUTPUT:',OUTPUT)


In [ ]:
STATIC=OUTPUT/'static_audits'/f'{ARM}_seed{SEED}.json'
STATIC.parent.mkdir(parents=True,exist_ok=True)
if ARM in {'AF2R0','AF2R1'}:
    from coffee_detector.af2r.audit import run_af2r_static_audit
    audit=run_af2r_static_audit(SOURCE,STATIC,device='cuda:0')
    assert audit['decision']=='PASS', audit['gates']
elif ARM=='AF2CAL3':
    from coffee_detector.af2cal.audit import run_af2cal_static_audit
    audit=run_af2cal_static_audit(SOURCE,STATIC,device='cuda:0')
    assert audit['decision']=='PASS', audit['gates']
else:
    print('FCT0 memakai checkpoint-invariance audit setelah training.')
print('STATIC READY:',STATIC if STATIC.exists() else 'post-train audit')


In [ ]:
import csv, time
LOGS=OUTPUT/'logs'; LOGS.mkdir(parents=True,exist_ok=True)
LOG=LOGS/f'{ARM}_seed{SEED}_run.log'
if ARM=='FCT0':
    module='coffee_detector.experiments.run_faruq_v3_fct0_confirmation_arm'
    extra=['--stb-confirmation',str(STB_CONFIRM),'--stb-checkpoint',str(SOURCE)]
    run_dir=OUTPUT/f'FCT0_seed{SEED}'
elif ARM in {'AF2R0','AF2R1'}:
    module='coffee_detector.experiments.run_faruq_v3_af2r_arm'
    extra=['--arm',ARM,'--af2-checkpoint',str(SOURCE),'--static-audit',str(STATIC)]
    run_dir=OUTPUT/ARM/f'{ARM}_seed{SEED}'
else:
    module='coffee_detector.experiments.run_faruq_v3_af2cal_arm'
    extra=['--arm',ARM,'--af2-checkpoint',str(SOURCE),'--static-audit',str(STATIC)]
    run_dir=OUTPUT/ARM/f'{ARM}_seed{SEED}'
command=[sys.executable,'-u','-m',module,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--output-root',str(OUTPUT),'--seed',str(SEED),'--device','0','--authorize-training',*extra]
print('START/RESUME:',ARM,SEED,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    shown=-1
    while process.poll() is None:
        csv_path=run_dir/'results.csv'
        epochs=0
        if csv_path.is_file():
            try:
                with csv_path.open(newline='',encoding='utf-8') as handle: epochs=len(list(csv.DictReader(handle)))
            except Exception: pass
        if epochs!=shown: print(f'{ARM} seed {SEED}: {epochs} epoch tercatat',flush=True); shown=epochs
        time.sleep(30)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:]))
    raise RuntimeError(f'{ARM} seed {SEED} gagal: {process.returncode}')
RESULT=OUTPUT/'val_reports'/f'{ARM}_seed{SEED}_result.json'
assert RESULT.is_file(), RESULT
print('SELESAI:',RESULT)


In [ ]:
import json
result=json.loads(RESULT.read_text())
metrics=result['metrics']
print({key:metrics[key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
print('TRAINING THIS CALL:',result.get('training_executed_this_call',result.get('training_executed')))
print('TEST:',result['test_images_accessed'])


## Keputusan agregat

Jalankan sel berikut hanya setelah kedelapan result seed 123/2026 tersedia.


In [ ]:
from coffee_detector.experiments.run_faruq_v3_top_controls_confirmation_decision import run_faruq_v3_top_controls_confirmation_decision
def pair(arm): return tuple(OUTPUT/'val_reports'/f'{arm}_seed{seed}_result.json' for seed in (123,2026))
required=[*pair('FCT0'),*pair('AF2R0'),*pair('AF2R1'),*pair('AF2CAL3')]
missing=[str(path) for path in required if not path.is_file()]
if missing:
    print('BELUM LENGKAP:',*missing,sep='\n- ')
else:
    decision=run_faruq_v3_top_controls_confirmation_decision(
        STB_CONFIRM,AF2_CONFIRM,CONT_CONFIRM,
        require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-fcstb-distillation-v1/val_reports/FCT0_seed42_val.json'),
        REPO/'docs/evidence/FARUQ_V3_AF2R_SCREENING_2026-08-17.json',
        REPO/'docs/evidence/FARUQ_V3_AF2_CHANNEL_CALIBRATION_SCREENING_2026-08-17.json',
        pair('FCT0'),pair('AF2R0'),pair('AF2R1'),pair('AF2CAL3'),
        OUTPUT/'val_reports/top_controls_paired_confirmation.json')
    print('RETAINED:',decision['retained'])
    for arm,row in decision['comparisons'].items(): print(arm,row['decision'],row['criteria'])
    print('SUMMARY:',decision['summary'])
